In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import src.config as config

In [3]:
# Define the current date and date range for the historical data
from datetime import datetime, timedelta
import pandas as pd

current_date= pd.to_datetime(datetime.utcnow()).floor("h")
print(f"{current_date= }")

# we fetch raw data for the last 28 days, to add redundancy to the data pipeline
fetch_data_to= current_date
fetch_data_from= current_date - timedelta(days=28)

current_date= Timestamp('2026-11-10 03:00:00')


C:\Users\Nima Mafi\AppData\Local\Temp\ipykernel_1056\3542526114.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_date= pd.to_datetime(datetime.utcnow()).floor("h")


In [4]:
# fetching recent data, but we have no access to NYC association data warehouses.
# instead, we simulate a call to a data warehouse

from src.data import load_raw_data

def fetch_batch_raw_data(from_data: datetime, to_date: datetime) -> pd.DataFrame:
    """
    simulate production data by sampling historical data from 52 weeks ago (i.e. 1 year ago)
    """
    # move back the date range by 1 year
    from_date_= from_data - timedelta(days= 7*52)
    to_date_= to_date - timedelta(days= 7*52)

    # dowanload 2 files from the data source
    rides= load_raw_data(year= from_date_.year, months= from_date_.month)
    rides= rides[rides.pickup_datetime >= from_date_]
    rides2= load_raw_data(year= to_date_.year, months= to_date_.month)
    rides2= rides2[rides2.pickup_datetime < to_date_]

    # shift bach the data 1 year ahead and pretend it is recent data
    rides["pickup_datetime"] += timedelta(days= 7*52)
    rides.sort_values(by=["pickup_location_id", "pickup_datetime"], inplace=True)

    return rides


In [5]:
rides= fetch_batch_raw_data(from_data= fetch_data_from, to_date= fetch_data_to)

File 2025-10 was already in local storage
File 2025-11 was already in local storage


In [7]:
# transform the raw data into time-series data
from src.data import transform_raw_data_into_ts_data
ts_data= transform_raw_data_into_ts_data(rides)
ts_data["pickup_location_id"] = ts_data["pickup_location_id"].astype("int64")

100%|██████████| 261/261 [00:00<00:00, 296.51it/s]


In [8]:
# connect to the feature store and feature group where we want to save the data
import hopsworks

# connect to the project
project= hopsworks.login(
    project= config.HOPSWORKS_PROJECT_NAME,
    api_key_value= config.HOPSWORKS_API_KEY
)

# connect to the feature store
feature_store= project.get_feature_store()

# connect to the feature group 
feature_group= feature_store.get_or_create_feature_group(
    name= config.FEATURE_GROUP_NAME,
    version= config.FEATURE_GROUP_VERSION,
    description= "Time-series data at hourly frequency",
    primary_key= ["pickup_hour", "pickup_location_id"],
    event_time= "pickup_hour",
    time_travel_format="HUDI",
)



2026-11-10 14:50:13,853 INFO: Initializing external client
2026-11-10 14:50:13,854 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/55417


In [9]:
# insert the data into the feature group
feature_group.insert(ts_data, write_options={"wait_for_job": False})

2026-11-10 14:50:18,696 INFO: Python Engine initialized.
Uploading Dataframe: 100.00% |██████████| Rows 111969/111969 | Elapsed Time: 00:15 | Remaining Time: 00:00


Launching job: time_series_hourly_feature_group_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/55417/jobs/named/time_series_hourly_feature_group_1_offline_fg_materialization/executions


(Job('time_series_hourly_feature_group_1_offline_fg_materialization', 'SPARK'),
 None)